# 01 - Simulate a paginated Python REST API

Run all cells in the existing OpenShift Jupyter environment. This notebook creates a small read-only API using Python's standard library. There are no Python packages to install and no helper files to import.

**Keep this kernel running while executing notebook 02 in another kernel.** The API runs on a background thread. Closing or restarting this kernel stops it. Notebook 03 uses stored Iceberg data and no longer needs the API.

The server binds port 18717 in the existing Spark/Jupyter Pod. Its headless `spark-master` DNS name lets Spark workers reach it. No public Route or new deployment is created. This is a workshop simulation with no authentication or TLS.

In [ ]:
import os
import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs
from urllib.request import urlopen
from urllib.error import HTTPError

PORT = 18717
API_BASE = os.getenv("REST_API_BASE_URL", f"http://spark-master:{PORT}").rstrip("/")
VERSION = "seed-v1"
DATA = {
    "products": [
        {"product_id":101,"product_name":"Keyboard, Compact","category":"Accessories","list_price":"25.00"},
        {"product_id":102,"product_name":"Wireless Mouse","category":"Accessories","list_price":"15.00"},
        {"product_id":103,"product_name":"USB Cable","category":"Connectivity","list_price":"10.00"},
        {"product_id":104,"product_name":"Docking Station","category":None,"list_price":"50.00"}],
    "customers": [
        {"customer_id":1,"customer_name":"Asha","country":"IN","email":"asha@example.test"},
        {"customer_id":2,"customer_name":"Ravi","country":"IN","email":None},
        {"customer_id":3,"customer_name":"Emma","country":"UK","email":"emma@example.test"},
        {"customer_id":4,"customer_name":"Liam","country":"US","email":"liam@example.test"}],
    "orders": [
        {"order_id":2001,"customer_id":1,"product_id":101,"quantity":2,"unit_price":"25.00","country":"IN"},
        {"order_id":2002,"customer_id":2,"product_id":102,"quantity":1,"unit_price":"15.00","country":"IN"},
        {"order_id":2003,"customer_id":3,"product_id":103,"quantity":3,"unit_price":"10.00","country":"UK"},
        {"order_id":2004,"customer_id":4,"product_id":104,"quantity":1,"unit_price":"50.00","country":"US"},
        {"order_id":2005,"customer_id":1,"product_id":103,"quantity":2,"unit_price":"10.00","country":"IN"},
        {"order_id":2006,"customer_id":3,"product_id":101,"quantity":1,"unit_price":"25.00","country":"UK"}]
}

## Start the server
The server exposes /health and /v1/products, /v1/customers and /v1/orders. Money is serialized as a string so the Spark transformation can cast it explicitly to DECIMAL.

In [ ]:
class WorkshopAPI(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass  # Keep notebook output focused on the requests we demonstrate.

    def reply(self, status, payload):
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        request = urlparse(self.path)
        if request.path == "/health":
            return self.reply(200, {"status":"ok", "dataset_version":VERSION})
        entity = request.path.removeprefix("/v1/")
        if not request.path.startswith("/v1/") or entity not in DATA:
            return self.reply(404, {"error":"Unknown endpoint"})
        query = parse_qs(request.query, keep_blank_values=True)
        if set(query) - {"page", "page_size", "country"}:
            return self.reply(400, {"error":"Unsupported query parameter"})
        try:
            page = int(query.get("page", ["1"])[0])
            size = int(query.get("page_size", ["2"])[0])
            if page < 1 or not 1 <= size <= 100:
                raise ValueError()
        except ValueError:
            return self.reply(400, {"error":"page must be positive; page_size must be 1..100"})
        country = query.get("country", [None])[0]
        if country is not None and (entity != "orders" or country not in {"IN","UK","US"}):
            return self.reply(400, {"error":"Use an orders country filter: IN, UK or US"})
        rows = DATA[entity]
        if country is not None:
            rows = [row for row in rows if row["country"] == country]
        start = (page - 1) * size
        return self.reply(200, {"dataset_version":VERSION,"entity":entity,"page":page,
            "total":len(rows),"next_page":page+1 if start+size < len(rows) else None,
            "items":rows[start:start+size]})

# Restart only the server object owned by this kernel when rerunning the notebook.
if "server" in globals():
    server.shutdown()
    server.server_close()
    server_thread.join(timeout=5)
server = ThreadingHTTPServer(("0.0.0.0", PORT), WorkshopAPI)
server.daemon_threads = True
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print("API listening:", API_BASE)

## Inspect a page and verify the API contract

In [ ]:
with urlopen(API_BASE + "/health", timeout=5) as response:
    assert json.load(response) == {"status":"ok","dataset_version":VERSION}
with urlopen(API_BASE + "/v1/orders?page=1&page_size=2", timeout=5) as response:
    first_page = json.load(response)
assert len(first_page["items"]) == 2 and first_page["next_page"] == 2 and first_page["total"] == 6
print(json.dumps(first_page, indent=2))
for entity, expected in [("products",4),("customers",4),("orders",6)]:
    with urlopen(API_BASE + f"/v1/{entity}?page_size=100", timeout=5) as response:
        assert len(json.load(response)["items"]) == expected
for path, expected in [("/missing",404),("/v1/orders?page=0",400),("/v1/orders?country=XX",400)]:
    try:
        urlopen(API_BASE + path, timeout=5)
        raise AssertionError("Expected HTTP error")
    except HTTPError as error:
        assert error.code == expected
print("PASS: API health, pagination, entity counts, 400 and 404 responses")

## Continue to notebook 02

Leave this kernel alive. Open notebook 02 in a separate Python kernel in the same Jupyter environment. Run notebook 02, stop its Spark session, then run notebook 03 in a fresh kernel.

After notebook 02 has completed and the API is no longer needed, stop this server manually in this kernel:

```python
server.shutdown()
server.server_close()
server_thread.join(timeout=5)
```

Run notebook 01 again to restart the simulated API. If the port is occupied by a different kernel, stop that known workshop server first; do not terminate unrelated processes.